# Lab 3: Hello, CUDA!
Chọn **Runtime → Change runtime type → T4 GPU**, rồi **Run all**.
Có thể tải `image.jpg` vào Files trước khi chạy; nếu không có, notebook dùng ảnh mẫu có sẵn.

Kết quả: ảnh, `metrics.csv`, báo cáo LaTeX và file `outputs/Lab3.zip`.
Thời gian GPU đã warm-up, đồng bộ; không tính truyền dữ liệu. Dùng GPU thật để đo speedup.


In [ ]:
%pip -q install "numba-cuda[cu12]"


In [ ]:
import time
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from numba import cuda, float32, float64

OUT = Path('outputs/Lab3')
OUT.mkdir(parents=True, exist_ok=True)

assert cuda.is_available(), 'Chọn Runtime > Change runtime type > T4 GPU rồi chạy lại.'
device = cuda.get_current_device()
print('GPU:', device.name)
MAX_THREADS = device.MAX_THREADS_PER_BLOCK
BLOCKS = [b for b in [(8, 8), (16, 16), (32, 16), (32, 32)]
          if b[0] * b[1] <= MAX_THREADS]
REPEATS = 3

def load_image(path='image.jpg'):
    if Path(path).exists():
        a = plt.imread(path)
        if np.issubdtype(a.dtype, np.floating):
            a = np.clip(a * 255, 0, 255).astype(np.uint8)
        return np.asarray(Image.fromarray(a).convert('RGB')).copy()
    # Ảnh mẫu có màu, cạnh và nhiễu; không cần tải file để Run all.
    y, x = np.indices((256, 256))
    a = np.stack((x, y, (x // 32 % 2) * 160 + 40), axis=-1)
    noise = np.random.default_rng(0).integers(-15, 16, a.shape)
    return np.clip(a + noise, 0, 255).astype(np.uint8)

image = load_image()
H, W = image.shape[:2]
gray = (0.299 * image[:, :, 0] + 0.587 * image[:, :, 1]
        + 0.114 * image[:, :, 2]).astype(np.uint8)
print('Ảnh:', image.shape)

def grid(block):
    return ((W + block[0] - 1) // block[0],
            (H + block[1] - 1) // block[1])

def gpu_time(run):
    run()  # Warm-up: không tính biên dịch JIT vào thời gian.
    cuda.synchronize()
    start = time.time()
    for _ in range(REPEATS):
        run()
    cuda.synchronize()
    return (time.time() - start) / REPEATS

def cpu_time(run):
    start = time.time()
    result = run()
    return result, time.time() - start

def show_images(items):
    fig, axes = plt.subplots(1, len(items), figsize=(4 * len(items), 4), squeeze=False)
    for ax, (title, a) in zip(axes[0], items):
        ax.imshow(a, cmap='gray', vmin=0, vmax=255)
        ax.set_title(title)
        ax.axis('off')
    fig.tight_layout()
    fig.savefig(OUT / 'images.png', dpi=120)
    plt.show()

def save_image(name, a):
    Image.fromarray(a).save(OUT / name)

def save_results(rows, report_name, report_text):
    df = pd.DataFrame(rows)
    df.to_csv(OUT / 'metrics.csv', index=False)
    print(df.to_string(index=False))
    # Bảng LaTeX nhỏ, không cần thêm thư viện dựng báo cáo.
    def tex(v):
        return str(v).replace('_', r'\_')
    lines = [r'\begin{center}', r'\resizebox{\linewidth}{!}{',
             r'\begin{tabular}{' + 'l' * len(df.columns) + '}', r'\hline',
             ' & '.join(tex(c) for c in df.columns) + r' \\', r'\hline']
    for row in df.itertuples(index=False, name=None):
        lines.append(' & '.join(f'{v:.6g}' if isinstance(v, float) else tex(v)
                                for v in row) + r' \\')
    lines += [r'\hline', r'\end{tabular}', '}', r'\end{center}']
    (OUT / 'results.tex').write_text('\n'.join(lines), encoding='utf-8')
    (OUT / report_name).write_text(report_text, encoding='utf-8')
    shutil.make_archive(str(OUT), 'zip', OUT)
    print('Kết quả và báo cáo:', str(OUT) + '.zip (tải trong Files bên trái)')


In [ ]:
def gray_cpu(src):
    out = np.empty(src.shape[:2], np.uint8)
    for y in range(src.shape[0]):
        for x in range(src.shape[1]):
            r, g, b = src[y, x]
            out[y, x] = int(0.299 * r + 0.587 * g + 0.114 * b)
    return out

@cuda.jit
def gray_1d(src, dst):
    i = cuda.grid(1)
    if i < dst.size:
        dst[i] = int(0.299 * src[i, 0] + 0.587 * src[i, 1] + 0.114 * src[i, 2])

@cuda.jit
def gray_2d(src, dst):
    x, y = cuda.grid(2)
    if x < src.shape[1] and y < src.shape[0]:
        dst[y, x] = int(0.299 * src[y, x, 0] + 0.587 * src[y, x, 1]
                        + 0.114 * src[y, x, 2])


In [ ]:
reference, cpu_s = cpu_time(lambda: gray_cpu(image))
d_src = cuda.to_device(image.reshape(-1, 3))
d_dst = cuda.device_array(H * W, dtype=np.uint8)
rows = []
for block in [32, 64, 128, 256, 512]:
    if block > MAX_THREADS:
        continue
    blocks = (H * W + block - 1) // block
    seconds = gpu_time(lambda: gray_1d[blocks, block](d_src, d_dst))
    result = d_dst.copy_to_host().reshape(H, W)
    error = int(np.abs(result.astype(int) - reference.astype(int)).max())
    assert error <= 1
    rows.append(dict(block=block, cpu_ms=cpu_s * 1000,
                     gpu_ms=seconds * 1000, speedup=cpu_s / seconds, max_error=error))
save_image('image_gray_cpu.png', reference)
save_image('image_gray_gpu.png', result)
show_images([('CPU', reference), ('GPU 1D', result)])
plt.figure(figsize=(6, 3))
plt.plot([r['block'] for r in rows], [r['gpu_ms'] for r in rows], 'o-')
plt.xlabel('Threads / block'); plt.ylabel('GPU time (ms)'); plt.grid()
plt.tight_layout(); plt.savefig(OUT / 'performance.png', dpi=120); plt.show()


In [ ]:
report_text = r"""
\documentclass{article}
\usepackage[utf8]{inputenc}
\usepackage[T5]{fontenc}
\usepackage{graphicx}
\usepackage[margin=1.5cm]{geometry}
\title{Lab 3: Hello, CUDA!}
\author{Nguyễn Thanh Hùng -- 2540056}
\date{}
\begin{document}
\maketitle
Each CPU iteration and each GPU thread computes $Y=0.299R+0.587G+0.114B$.
The RGB image is flattened to $(HW,3)$ for the 1D kernel. Blocks of 32--512 threads are compared.
The notebook shows both images, checks the maximum pixel error, and plots block size versus GPU time.
Select the fastest measured block; larger blocks do not necessarily run faster.

Kernel timing uses \texttt{time.time()} after JIT warm-up and CUDA synchronization,
averaged over three GPU runs. Device allocation and host/device copies are excluded.
CPU measurements are single runs. Speedup is relative to the stated CPU implementation.
Actual measurements appear after running the notebook on Colab; no timings are assumed here.
\IfFileExists{results.tex}{\input{results.tex}}{}
\IfFileExists{performance.png}{\begin{center}\includegraphics[width=.75\linewidth]{performance.png}\end{center}}{}
\IfFileExists{images.png}{\begin{center}\includegraphics[width=\linewidth]{images.png}\end{center}}{}
\IfFileExists{histograms.png}{\begin{center}\includegraphics[width=.8\linewidth]{histograms.png}\end{center}}{}
\end{document}
"""
save_results(rows, 'Report.3.cuda.tex', report_text)
